In [1]:
import numpy as np
import pandas as pd
from configs import (TABLES, ROOT, HEAVY, MIN_GENES, DROP_CONTEXTS,
                     DROP_RELIABILITY, POSITIVE, NEGATIVE, SPLIT_FRACTIONS)

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from utils import metrics, within_auc, within_auc_parts
import socket
import time

In [3]:
SPLIT = "split_pair"
# rank and vector penalty are chosen on validation pairs from these grids
RANKS = [2, 4, 8, 16, 32]
LAMBDA_VEC_GRID = [1e-7, 1e-6, 1e-5, 1e-4]
SEEDS = [1, 2, 3]
# vectors start small and random: at zero, each vector's gradient is the other vector, also zero, so neither would move
INIT_SD = 0.05
# optimiser settings, fixed rather than tuned; early stopping on validation log loss sets the effective training length
LR = 0.05
BATCH_SIZE = 4096
EPOCHS = 60
PATIENCE = 8
N_BOOT = 2000

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
machine = socket.gethostname()
torch.use_deterministic_algorithms(True, warn_only=True)
print("device", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "", "machine", machine, "torch", torch.__version__)

device cuda NVIDIA RTX 4500 Ada Generation machine DESKTOP-2HRT079 torch 2.14.0+cu130


In [5]:
labels = pd.read_parquet(HEAVY / "01_labels.parquet")

In [6]:
labels

,pair_id,gene,tissue,cell_type,gene_name,y,context,split_pair,split_gene,split_context
0,0,ENSG00000000003,Adipose tissue,adipocytes,TSPAN6,0,Adipose tissue | adipocytes,train,train,train
1,1,ENSG00000000003,Adrenal gland,glandular cells,TSPAN6,0,Adrenal gland | glandular cells,train,train,train
2,2,ENSG00000000003,Appendix,glandular cells,TSPAN6,1,Appendix | glandular cells,train,train,train
3,3,ENSG00000000003,Appendix,lymphoid tissue,TSPAN6,0,Appendix | lymphoid tissue,train,train,train
4,4,ENSG00000000003,Bone marrow,hematopoietic cells,TSPAN6,0,Bone marrow | hematopoietic cells,train,train,val
...,...,...,...,...,...,...,...,...,...,...
784295,784295,ENSG00000291316,Tonsil,germinal center cells,ENSG00000291316,1,Tonsil | germinal center cells,train,train,train
784296,784296,ENSG00000291316,Tonsil,non-germinal center cells,ENSG00000291316,1,Tonsil | non-germinal center cells,train,train,val
784297,784297,ENSG00000291316,Tonsil,squamous epithelial cells,ENSG00000291316,1,Tonsil | squamous epithelial cells,train,train,test
784298,784298,ENSG00000291316,Urinary bladder,urothelial cells,ENSG00000291316,1,Urinary bladder | urothelial cells,train,train,train


In [9]:
genes = np.sort(labels["gene"].unique())
contexts = np.sort(labels["context"].unique())
labels["g"] = np.searchsorted(genes, labels["gene"])
labels["c"] = np.searchsorted(contexts, labels["context"])

In [16]:
parts = {name: labels[labels[SPLIT] == name].reset_index(drop=True) for name in ["train", "val", "test"]}
parts

{'train':         pair_id             gene           tissue                  cell_type  \
 0             0  ENSG00000000003   Adipose tissue                 adipocytes   
 1             1  ENSG00000000003    Adrenal gland            glandular cells   
 2             2  ENSG00000000003         Appendix            glandular cells   
 3             3  ENSG00000000003         Appendix            lymphoid tissue   
 4             4  ENSG00000000003      Bone marrow        hematopoietic cells   
 ...         ...              ...              ...                        ...   
 627435   784295  ENSG00000291316           Tonsil      germinal center cells   
 627436   784296  ENSG00000291316           Tonsil  non-germinal center cells   
 627437   784297  ENSG00000291316           Tonsil  squamous epithelial cells   
 627438   784298  ENSG00000291316  Urinary bladder           urothelial cells   
 627439   784299  ENSG00000291316           Vagina  squamous epithelial cells   
 
               ge

In [14]:
data = {
    name: (
        torch.tensor(df["g"].to_numpy(), device=device),
        torch.tensor(df["c"].to_numpy(), device=device),
        torch.tensor(df["y"].to_numpy(), dtype=torch.float32, device=device),
    )
    for name, df in parts.items()
}

In [15]:
data

{'train': (tensor([    0,     0,     0,  ..., 11154, 11154, 11154], device='cuda:0'),
  tensor([ 0,  1,  2,  ..., 72, 73, 74], device='cuda:0'),
  tensor([0., 0., 1.,  ..., 1., 1., 1.], device='cuda:0')),
 'val': (tensor([    0,     0,     0,  ..., 11154, 11154, 11154], device='cuda:0'),
  tensor([ 9, 19, 25,  ..., 43, 46, 62], device='cuda:0'),
  tensor([0., 0., 1.,  ..., 1., 0., 1.], device='cuda:0')),
 'test': (tensor([    0,     0,     0,  ..., 11154, 11154, 11154], device='cuda:0'),
  tensor([14, 56, 64,  ..., 41, 45, 65], device='cuda:0'),
  tensor([0., 0., 0.,  ..., 1., 1., 0.], device='cuda:0'))}

In [17]:
class MF(nn.Module):
    def __init__(self, n_genes, n_contexts, rank, init_sd):
        super().__init__()
        self.rank = rank
        self.intercept = nn.Parameter(torch.zeros(1))
        self.gene_bias = nn.Embedding(n_genes, 1)
        self.context_bias = nn.Embedding(n_contexts, 1)
        nn.init.zeros_(self.gene_bias.weight)
        nn.init.zeros_(self.context_bias.weight)
        if rank > 0:
            self.gene_vec = nn.Embedding(n_genes, rank)
            self.context_vec = nn.Embedding(n_contexts, rank)
            nn.init.normal_(self.gene_vec.weight, std=init_sd)
            nn.init.normal_(self.context_vec.weight, std=init_sd)

    def forward(self, g, c):
        logit = self.intercept + self.gene_bias(g).squeeze(1) + self.context_bias(c).squeeze(1)
        if self.rank > 0:
            logit = logit + (self.gene_vec(g) * self.context_vec(c)).sum(dim=1)
        return logit

    def penalties(self):
        bias = (self.gene_bias.weight ** 2).sum() + (self.context_bias.weight ** 2).sum()
        if self.rank == 0:
            return bias, torch.zeros((), device=bias.device)
        vec = (self.gene_vec.weight ** 2).sum() + (self.context_vec.weight ** 2).sum()
        return bias, vec


def predict(model, part):
    g, c, _ = data[part]
    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model(g, c)).cpu().numpy()


def score(model, part):
    return metrics(parts[part]["y"].to_numpy(), predict(model, part), parts[part]["g"].to_numpy(), parts[part]["c"].to_numpy())


def train(rank, lam_vec, seed, init_sd=INIT_SD, epochs=EPOCHS):
    torch.manual_seed(seed)
    gen = torch.Generator(device=device).manual_seed(seed)
    model = MF(len(genes), len(contexts), rank, init_sd).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    g_tr, c_tr, y_tr = data["train"]

    best, best_epoch, best_state, history = np.inf, 0, None, []
    for epoch in range(1, epochs + 1):
        model.train()
        perm = torch.randperm(n_train, device=device, generator=gen)
        batch_losses = []
        for start in range(0, n_train, BATCH_SIZE):
            idx = perm[start:start + BATCH_SIZE]
            bias_pen, vec_pen = model.penalties()
            loss = (
                F.binary_cross_entropy_with_logits(model(g_tr[idx], c_tr[idx]), y_tr[idx])
                + 0.5 * LAMBDA_BIAS * bias_pen + 0.5 * lam_vec * vec_pen
            )
            opt.zero_grad()
            loss.backward()
            opt.step()
            batch_losses.append(loss.item())
        lr_now = sched.get_last_lr()[0]
        sched.step()

        val = score(model, "val")
        history.append({"epoch": epoch, "lr": lr_now, "train_objective": float(np.mean(batch_losses)), **{f"val_{k}": v for k, v in val.items()}})
        if val["log_loss"] < best:
            best, best_epoch = val["log_loss"], epoch
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        elif epoch - best_epoch >= PATIENCE:
            break

    model.load_state_dict(best_state)
    return model, best_epoch, epoch, history

In [25]:
LAMBDA_BIAS = float(pd.read_csv(TABLES / "02_additive_summary.csv")["lambda"].iloc[0])
n_train = len(parts["train"])
settings = [(0, 0.0)] + [(r, lam) for r in RANKS for lam in LAMBDA_VEC_GRID]
runs, histories, states = [], [], {}
t_grid = time.perf_counter()
for rank, lam_vec in settings:
    for seed in SEEDS:
        t0 = time.perf_counter()
        model, best_epoch, last_epoch, history = train(rank, lam_vec, seed)
        val = score(model, "val")
        runs.append({"rank": rank, "lambda_vec": lam_vec, "seed": seed, "best_epoch": best_epoch, "epochs_run": last_epoch,
                     "seconds": time.perf_counter() - t0, **{f"val_{k}": v for k, v in val.items()}})
        histories += [{"rank": rank, "lambda_vec": lam_vec, "seed": seed, **h} for h in history]
        states[(rank, lam_vec, seed)] = {k: v.cpu() for k, v in model.state_dict().items()}
    r = pd.DataFrame(runs[-len(SEEDS):])

runs = pd.DataFrame(runs).assign(machine=machine, device=str(device))


by_setting = runs.groupby(["rank", "lambda_vec"])["val_auc_within_gene"].mean()
best_rank, best_lam = by_setting.drop(index=0, level="rank").idxmax()

In [26]:
best_rank

np.int64(8)

In [27]:
y_te, g_te, c_te = (parts["test"][k].to_numpy() for k in ["y", "g", "c"])
test_rows, preds, u_parts = [], [], {}
for rank, lam_vec in [(0, 0.0), (best_rank, best_lam)]:
    label = "additive" if rank == 0 else "mf"
    for seed in SEEDS:
        model = MF(len(genes), len(contexts), rank, INIT_SD).to(device)
        model.load_state_dict(states[(rank, lam_vec, seed)])
        for part in ["val", "test"]:
            preds.append(pd.DataFrame({"pair_id": parts[part]["pair_id"], "part": part, "model": label, "seed": seed, "p": predict(model, part)}))
        p_te = preds[-1]["p"].to_numpy()
        test_rows.append({"model": label, "rank": rank, "lambda_vec": lam_vec, "seed": seed, **{f"test_{k}": v for k, v in metrics(y_te, p_te, g_te, c_te).items()}})
        u_parts[(label, seed)] = within_auc_parts(y_te, p_te, g_te)

test = pd.DataFrame(test_rows)

In [28]:
test

,model,rank,lambda_vec,seed,test_log_loss,test_auc,test_auc_within_gene,test_auc_within_context
0,additive,0,0.00000,1,0.333415,0.931439,0.742527,0.924062
1,additive,0,0.00000,2,0.333149,0.931610,0.743668,0.924142
2,additive,0,0.00000,3,0.333503,0.931409,0.743793,0.924017
3,mf,8,0.00001,1,0.287137,0.949208,0.825200,0.943330
4,mf,8,0.00001,2,0.286659,0.949533,0.826388,0.943760
5,mf,8,0.00001,3,0.286832,0.949335,0.825825,0.943467
